In [ ]:
# ── Installs ──────────────────────────────────────────────────────────────────
!pip install opencv-python-headless anthropic -q
!pip install opencv-python-headless anthropic tqdm -q

# ── Imports ───────────────────────────────────────────────────────────────────
import os
import json
import base64
import cv2
import numpy as np
import pandas as pd
import anthropic
from sklearn.metrics import (
    log_loss, accuracy_score, f1_score,
    precision_score, recall_score, confusion_matrix
)
from dotenv import dotenv_values
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
config = dotenv_values("../.env")
os.environ["ANTHROPIC_API_KEY"] = config["ANTHROPIC_API_KEY"]
client = anthropic.Anthropic()
BASE_DIR = "/Users/kaatsandoval/code/katiarojas87/final-project"

LUXURY_THRESHOLD     = 0.55
BRIGHTNESS_THRESHOLD = 0.65
CONDITION_THRESHOLD  = 0.65
SPACIOUSNESS_THRESHOLD  = 0.65


# ── Data ──────────────────────────────────────────────────────────────────────
df = pd.read_pickle("pw")
df["image_path"] = df["image_path"].apply(
    lambda p: p if os.path.isabs(p) else os.path.join(BASE_DIR, p)
)


# ── Claude Vision: luxury + condition scores (0–1) ────────────────────────────
VISION_PROMPT = """Assess this room image. Reply ONLY with valid JSON, no markdown:
{
  "luxury": <float 0.0 to 1.0>,
  "condition": <float 0.0 to 1.0>,
  "brightness": <float 0.0 to 1.0>
  "spaciousness": <float 0.0 to 1.0>
}

Guidelines:
- luxury:
0.0–0.2 = builder-grade laminate, hollow-core doors, vinyl flooring, basic white fixtures, stock cabinets, no architectural detail,
0.3–0.5 = mid-range finishes, ceramic tile, standard granite counters, basic stainless appliances, simple crown molding,
0.6–0.8 = premium hardwood floors, quartz countertops, custom cabinetry, designer light fixtures, high-end appliances (Sub-Zero, Wolf),
0.9–1.0 = marble/travertine surfaces, bespoke millwork, coffered ceilings, statement chandelier, integrated smart home, herringbone parquet, designer fixtures (Waterworks, Restoration Hardware)

- condition:
condition:  0.0 = severe damage (peeling paint, water stains, cracked tiles, mold),
1.0 = pristine/immaculate (spotless, freshly renovated, zero blemishes)

- brightness:
0.0–0.2 = no windows visible, artificial light only, dim overhead bulb, dark corners, heavy blackout curtains,
0.3–0.5 = some natural light, partially shaded, small windows, moderate artificial lighting,
0.6–0.8 = good natural light, large windows, bright and airy feel, south-facing exposure,
0.9–1.0 = floor-to-ceiling windows, sun-drenched, flooded with natural light, skylights, panoramic glass, glowing interior

- spaciousness:
0.0–0.2 = cramped, cluttered, low ceiling, furniture blocking pathways, no circulation space, tight corridors,
0.3–0.5 = average room size, standard ceiling height, functional but not generous, modest proportions,
0.6–0.8 = open floor plan, generous proportions, good circulation, 10ft+ ceilings, minimal visual clutter,
0.9–1.0 = grand, voluminous, double-height ceilings, sweeping xopen plan, loft-like, unobstructed sightlines, expansive
"""


def assess_room_vision(image_path: str) -> dict:
    ext = image_path.lower().split(".")[-1]
    media_type = "image/jpeg" if ext in ("jpg", "jpeg") else f"image/{ext}"
    with open(image_path, "rb") as f:
        image_data = base64.b64encode(f.read()).decode("utf-8")

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=64,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": media_type, "data": image_data}},
                {"type": "text", "text": VISION_PROMPT}
            ]
        }]
    )

    raw = response.content[0].text.strip()
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(raw)

# ── Score all images ──────────────────────────────────────────────────────────
score_bright, score_lux, score_cond = [], [], []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Scoring images"):
    path = row["image_path"]

    try:
        result = assess_room_vision(path)
        score_bright.append(float(result["brightness"]))
        score_lux.append(float(result["luxury"]))
        score_cond.append(float(result["condition"]))
    except Exception as e:
        print(f"[{i}] Error: {e}")
        score_bright.append(None)
        score_lux.append(None)
        score_cond.append(None)

df["score_bright"] = score_bright
df["score_lux"]    = score_lux
df["score_cond"]   = score_cond

df.to_pickle("assessment_output_v3.pkl")
df.to_csv("scores_output.csv", index=False)

print("Done. Saved → assessment_output_v3.pkl + scores_output.csv")
print(df[["score_bright", "score_lux", "score_cond"]].describe())

# ── Metrics ───────────────────────────────────────────────────────────────────
EVAL_CONFIGS = [
    ("Luxury",     "score_lux",    lambda d: d["Luxury"] == "yes",                LUXURY_THRESHOLD),
    ("Brightness", "score_bright", lambda d: d["Brightness"] == "yes",            BRIGHTNESS_THRESHOLD),
    ("Condition",  "score_cond",   lambda d: d["Condition"].isin(["new", "yes"]), CONDITION_THRESHOLD),
]

df_eval = df[df["score_bright"] != -1000.0].dropna(subset=["score_lux", "score_cond"]).copy()

for label, score_col, y_true_fn, threshold in EVAL_CONFIGS:
    y_true = y_true_fn(df_eval).astype(int)
    y_prob = df_eval[score_col]
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    print(f"\n── {label.upper()} ──")
    print(f"  class_balance : {y_true.mean():.1%} positive")
    print(f"  log_loss      : {log_loss(y_true, y_prob):.4f}")
    print(f"  accuracy      : {accuracy_score(y_true, y_pred):.4f}")
    print(f"  precision     : {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"  recall        : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"  f1            : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"  confusion     : TN={tn} FP={fp} FN={fn} TP={tp}")

FileNotFoundError: [Errno 2] No such file or directory: 'assessment_output_v2.pkl'

In [5]:
config

Available objects for config:
    AliasManager
    DisplayFormatter
    HistoryManager
    IPCompleter
    IPKernelApp
    LoggingMagics
    MagicsManager
    OSMagics
    PrefilterManager
    ScriptMagics
    StoreMagics
    ZMQDisplayPublisher
    ZMQInteractiveShell
